# MACCS fingerprints with Data 1.1

## 1. Data ingestion and cleaning

In [ ]:
import pandas as pd

df_cdd = pd.read_csv('data_1.csv')
df_cdd = df_cdd.rename(columns={
    "GU 10 µM L6: Average Glucose uptake / isoprenaline (%)": "gu",
    "B2 cAMP L6: Average cAMP / isoprenaline (AU)": "camp",
})

In [ ]:
df_cdd = df_cdd.drop(['Synonyms',
                      'GU 10 µM L6: Average Glucose uptake / isoprenaline (%) Standard Deviation (±)',
                      'GU 10 µM L6: Average Glucose uptake / isoprenaline (%) Count',
                      'B2 cAMP L6: Average cAMP / isoprenaline (AU) Standard Deviation (±)',
                      'B2 cAMP L6: Average cAMP / isoprenaline (AU) Count'
                    ], axis=1)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from copy import deepcopy

# RDKit
from rdkit import Chem
from rdkit.Chem import MACCSkeys, Draw
from rdkit.Chem import rdMolDescriptors

# Sklearn
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

# XGBoost
import xgboost as xgb

# SHAP
import shap

# PyTorch & PyG
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import NNConv, global_mean_pool
from torch_geometric.explain import Explainer, GNNExplainer

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print('All imports successful.')

In [ ]:
df = df_cdd
df = df.dropna(subset=['camp'])
df.columns = df.columns.str.strip()

# Basic sanity check
required = {'Molecule Name', 'SMILES', 'gu', 'camp'}
assert required.issubset(df.columns), f'Missing columns: {required - set(df.columns)}'

print(f'Dataset shape: {df.shape}')
print(df[['gu','camp']].describe())
df.head()

## 2. MACCS Keys Fingerprints (167-bit)

> MACCS keys 1–166 are used (bit 0 is always 0 in RDKit).

In [ ]:
def smiles_to_maccs(smi):
    mol = Chem.MolFromSmiles(smi)
    fp = MACCSkeys.GenMACCSKeys(mol)
    return np.array(fp)

X = np.vstack(df['SMILES'].apply(smiles_to_maccs).values)  # (N, 167)
y = df[['gu', 'camp']].values  # (N, 2)

# MACCS bit names for later labelling
maccs_names = [f'MACCS_{i}' for i in range(167)]

# Remove always-zero or always-one bits (no variance)
var_mask = (X.var(axis=0) > 0)
X_filtered = X[:, var_mask]
names_filtered = [maccs_names[i] for i, v in enumerate(var_mask) if v]

print(f'MACCS shape (raw): {X.shape}')
print(f'MACCS shape (after removing zero-variance bits): {X_filtered.shape}')

# Visualise bit frequency
bit_freq = X_filtered.mean(axis=0)
plt.figure(figsize=(14,3))
plt.bar(range(len(bit_freq)), bit_freq, color='steelblue')
plt.xlabel('MACCS bit index (filtered)')
plt.ylabel('Frequency')
plt.title('MACCS Key Bit Frequencies across the Dataset')
plt.tight_layout()
plt.show()

In [ ]:
# Extract target variables
target_cols = ['gu', 'camp']
colors = ['steelblue', 'darkorange'] 

y = df[target_cols].values
print("Target variables shape:", y.shape)

# Visualize target distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes = axes.ravel()

for idx, col in enumerate(target_cols):
    axes[idx].hist(df[col], bins=30, edgecolor='black', alpha=0.7, color=colors[idx])
    axes[idx].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('target_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Cross validation

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

def cv_metrics(y_true, y_pred, target_names=['gu','camp']):
    """Return per-target RMSE, R2."""
    results = {}
    for i, name in enumerate(target_names):
        results[name] = {
            'RMSE': np.sqrt(mean_squared_error(y_true[:,i], y_pred[:,i])),
            'R2':   r2_score(y_true[:,i], y_pred[:,i])
        }
    return results

def print_cv_summary(all_fold_metrics, model_name):
    print(f'\n{'─'*55}')
    print(f'  {model_name} — 5-Fold CV Summary')
    print(f'{'─'*55}')
    for target in ['gu','camp']:
        rmse_vals = [m[target]['RMSE'] for m in all_fold_metrics]
        r2_vals   = [m[target]['R2']   for m in all_fold_metrics]
        print(f'  [{target.upper()}]  RMSE: {np.mean(rmse_vals):.4f}±{np.std(rmse_vals):.4f} | '
              f'R2: {np.mean(r2_vals):.4f}±{np.std(r2_vals):.4f}')
    print(f'{'─'*55}')

## 4. Machine Learning models 
### 4.1 Multi-output support vector regressor

In [ ]:
svr_fold_metrics = []
svr_fold_models  = []   # store (scaler, model) per fold

for fold, (train_idx, val_idx) in enumerate(kf.split(X_filtered)):
    X_tr, X_val = X_filtered[train_idx], X_filtered[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    scaler = StandardScaler()
    X_tr_sc  = scaler.fit_transform(X_tr)
    X_val_sc = scaler.transform(X_val)

    # MultiOutputRegressor wraps SVR for 2-target regression
    base_svr = SVR(kernel='rbf', C=10, epsilon=0.1, gamma='scale')
    mo_svr   = MultiOutputRegressor(base_svr, n_jobs=-1)
    mo_svr.fit(X_tr_sc, y_tr)

    y_pred = mo_svr.predict(X_val_sc)
    metrics = cv_metrics(y_val, y_pred)
    svr_fold_metrics.append(metrics)
    svr_fold_models.append((scaler, mo_svr, train_idx, val_idx))
    print(f'Fold {fold+1}: GU_RMSE={metrics["gu"]["RMSE"]:.4f}  cAMP_RMSE={metrics["camp"]["RMSE"]:.4f}')

print_cv_summary(svr_fold_metrics, 'Multi-Output SVR')

### 4.2 Multi-output random forest

In [ ]:
rf_fold_metrics = []
rf_fold_models  = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_filtered)):
    X_tr, X_val = X_filtered[train_idx], X_filtered[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    rf = RandomForestRegressor(
        n_estimators=300, max_features='sqrt',
        random_state=SEED, n_jobs=-1
    )
    # RandomForestRegressor natively handles multi-output
    rf.fit(X_tr, y_tr)

    y_pred = rf.predict(X_val)
    metrics = cv_metrics(y_val, y_pred)
    rf_fold_metrics.append(metrics)
    rf_fold_models.append((rf, train_idx, val_idx))
    print(f'Fold {fold+1}: GU_RMSE={metrics["gu"]["RMSE"]:.4f}  cAMP_RMSE={metrics["camp"]["RMSE"]:.4f}')

print_cv_summary(rf_fold_metrics, 'Multi-Output Random Forest')

### 4.3 Multi-output XGBoost

In [ ]:
xgb_fold_metrics = []
xgb_fold_models  = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_filtered)):
    X_tr, X_val = X_filtered[train_idx], X_filtered[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    # XGBoost ≥ 2.0 supports multi-output natively with multi_strategy
    xgb_model = xgb.XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        multi_strategy='multi_output_tree',   # native multi-output
        tree_method='hist',
        random_state=SEED,
        n_jobs=-1
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    y_pred = xgb_model.predict(X_val)
    metrics = cv_metrics(y_val, y_pred)
    xgb_fold_metrics.append(metrics)
    xgb_fold_models.append((xgb_model, train_idx, val_idx))
    print(f'Fold {fold+1}: GU_RMSE={metrics["gu"]["RMSE"]:.4f}  cAMP_RMSE={metrics["camp"]["RMSE"]:.4f}')

print_cv_summary(xgb_fold_metrics, 'Multi-Output XGBoost')

### 4.4 Model performance comparison

In [ ]:
def aggregate_cv(fold_metrics, model_name):
    rows = []
    for target in ['gu','camp']:
        for metric in ['RMSE','R2']:
            vals = [m[target][metric] for m in fold_metrics]
            rows.append({'Model': model_name, 'Target': target,
                        'Metric': metric, 'Mean': np.mean(vals), 'Std': np.std(vals)})
    return pd.DataFrame(rows)

results_df = pd.concat([
    aggregate_cv(svr_fold_metrics,  'SVR'),
    aggregate_cv(rf_fold_metrics,   'Random Forest'),
    aggregate_cv(xgb_fold_metrics,  'XGBoost'),
], ignore_index=True)

# Pivot for readability
pivot = results_df.pivot_table(index=['Model','Target'], columns='Metric', values=['Mean','Std'])
print(pivot.round(4))

# Bar chart with best model highlighted in green
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, target in zip(axes, ['gu', 'camp']):
    sub = results_df[(results_df['Target'] == target) & (results_df['Metric'] == 'RMSE')]

    # Identify best model (lowest RMSE)
    best_idx = sub['Mean'].idxmin()
    best_model = sub.loc[best_idx, 'Model']

    # Assign colors: green for best, blue for others
    colors = ['green' if m == best_model else 'steelblue' for m in sub['Model']]

    ax.bar(sub['Model'], sub['Mean'], yerr=sub['Std'], 
        capsize=5, color=colors, alpha=0.85)

    ax.set_title(f'{target.upper()} — Mean RMSE (5-Fold CV)')
    ax.set_ylabel('RMSE')
    ax.tick_params(axis='x', rotation=20)

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


print('\nExpected output: bar chart + pivot table with RMSE/MAE/R2 for all 4 models on both targets.')

### 4.5. Overfitting / underfitting diagnostics

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# OVERFITTING / UNDERFITTING DIAGNOSTICS
# Run AFTER Cells 14 (SVR), 16 (RF), and 18 (XGB) have been executed.
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import r2_score, mean_squared_error

TARGETS = ['gu', 'camp']
TARGET_LABELS = ['GU', 'cAMP']

# ─────────────────────────────────────────────────────────────────────────────
# 1. TRAIN vs. VALIDATION RMSE  (bias-variance gap)
# ─────────────────────────────────────────────────────────────────────────────
# Re-evaluate each stored fold model on its own TRAINING split

def compute_train_val_rmse(fold_models, model_type, X, y):
    """
    Returns two dicts keyed by target: lists of (train_rmse, val_rmse) per fold.
    model_type: 'svr' | 'rf' | 'xgb'
    """
    train_rmses = {t: [] for t in TARGETS}
    val_rmses   = {t: [] for t in TARGETS}

    for entry in fold_models:
        if model_type == 'svr':
            scaler, model, tr_idx, val_idx = entry
            X_tr  = scaler.transform(X[tr_idx])
            X_val = scaler.transform(X[val_idx])
        else:
            model, tr_idx, val_idx = entry
            X_tr  = X[tr_idx]
            X_val = X[val_idx]

        y_tr_pred  = model.predict(X_tr)
        y_val_pred = model.predict(X_val)

        for i, t in enumerate(TARGETS):
            train_rmses[t].append(np.sqrt(mean_squared_error(y[tr_idx, i],  y_tr_pred[:, i])))
            val_rmses[t].append(  np.sqrt(mean_squared_error(y[val_idx, i], y_val_pred[:, i])))

    return train_rmses, val_rmses


svr_tr_rmse, svr_val_rmse  = compute_train_val_rmse(svr_fold_models,  'svr', X_filtered, y)
rf_tr_rmse,  rf_val_rmse   = compute_train_val_rmse(rf_fold_models,   'rf',  X_filtered, y)
xgb_tr_rmse, xgb_val_rmse  = compute_train_val_rmse(xgb_fold_models,  'xgb', X_filtered, y)

model_results = {
    'SVR':           (svr_tr_rmse,  svr_val_rmse),
    'Random Forest': (rf_tr_rmse,   rf_val_rmse),
    'XGBoost':       (xgb_tr_rmse,  xgb_val_rmse),
}

folds = np.arange(1, 6)

fig, axes = plt.subplots(2, 3, figsize=(17, 9), sharey='row')
fig.suptitle('Train vs. Validation RMSE per Fold\n'
             '(Large gap = overfitting  |  Both high = underfitting)',
             fontsize=13, fontweight='bold')

for col, (mname, (tr_rmse, val_rmse)) in enumerate(model_results.items()):
    for row, t in enumerate(TARGETS):
        ax = axes[row, col]
        ax.plot(folds, tr_rmse[t],  'o-', color='steelblue',  label='Train RMSE', lw=2)
        ax.plot(folds, val_rmse[t], 's--', color='tomato',    label='Val RMSE',   lw=2)
        ax.fill_between(folds, tr_rmse[t], val_rmse[t], alpha=0.15, color='orange',
                        label='Gap (overfit zone)')
        ax.set_title(f'{mname}  [{TARGET_LABELS[row]}]')
        ax.set_xlabel('Fold')
        ax.set_ylabel('RMSE')
        ax.legend(fontsize=8)
        ax.set_xticks(folds)

plt.tight_layout()
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# 2. SUMMARY TABLE  — mean train RMSE vs. mean val RMSE + gap ratio
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd

rows = []
for mname, (tr_rmse, val_rmse) in model_results.items():
    for t in TARGETS:
        tr_mean  = np.mean(tr_rmse[t])
        val_mean = np.mean(val_rmse[t])
        gap      = val_mean - tr_mean
        ratio    = val_mean / tr_mean if tr_mean > 0 else np.nan

        if ratio > 2.0:
            diagnosis = '⚠️  Overfitting'
        elif val_mean > 0.5 * y[:, TARGETS.index(t)].std() * 2:
            diagnosis = '⚠️  Underfitting'
        else:
            diagnosis = '✅  Good fit'

        rows.append({
            'Model':          mname,
            'Target':         t,
            'Train RMSE':     round(tr_mean,  4),
            'Val RMSE':       round(val_mean, 4),
            'Gap (Val-Train)':round(gap,       4),
            'Val/Train ratio':round(ratio,     2),
            'Diagnosis':      diagnosis,
        })

summary_df = pd.DataFrame(rows)
print('\n── Overfitting / Underfitting Summary ──────────────────────────────────')
print(summary_df.to_string(index=False))


# ─────────────────────────────────────────────────────────────────────────────
# 3. PREDICTED vs. ACTUAL  (best fold of each model)
# ─────────────────────────────────────────────────────────────────────────────
def get_best_val_preds(fold_models, model_type, fold_metrics, X, y):
    best = np.argmin([(m['gu']['RMSE'] + m['camp']['RMSE']) / 2 for m in fold_metrics])
    entry = fold_models[best]
    if model_type == 'svr':
        scaler, model, tr_idx, val_idx = entry
        X_val = scaler.transform(X[val_idx])
    else:
        model, tr_idx, val_idx = entry
        X_val = X[val_idx]
    return y[val_idx], model.predict(X_val)

svr_yv, svr_yp  = get_best_val_preds(svr_fold_models,  'svr', svr_fold_metrics,  X_filtered, y)
rf_yv,  rf_yp   = get_best_val_preds(rf_fold_models,   'rf',  rf_fold_metrics,   X_filtered, y)
xgb_yv, xgb_yp  = get_best_val_preds(xgb_fold_models,  'xgb', xgb_fold_metrics,  X_filtered, y)

model_preds = [
    ('SVR',           svr_yv, svr_yp),
    ('Random Forest', rf_yv,  rf_yp),
    ('XGBoost',       xgb_yv, xgb_yp),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Predicted vs. Actual  (Best Fold)\n'
             'Points on the diagonal = perfect prediction',
             fontsize=13, fontweight='bold')

for col, (mname, yv, yp) in enumerate(model_preds):
    for row, (ti, tname) in enumerate(zip(range(2), TARGET_LABELS)):
        ax = axes[row, col]
        ax.scatter(yv[:, ti], yp[:, ti], alpha=0.6, edgecolors='k', linewidths=0.3,
                   color='steelblue', s=40)
        lo = min(yv[:, ti].min(), yp[:, ti].min())
        hi = max(yv[:, ti].max(), yp[:, ti].max())
        ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Ideal')
        r2 = r2_score(yv[:, ti], yp[:, ti])
        ax.set_title(f'{mname}  [{tname}]  R²={r2:.3f}')
        ax.set_xlabel('Actual')
        ax.set_ylabel('Predicted')
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# 4. RESIDUAL PLOTS  (best fold)
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Residuals vs. Predicted  (Best Fold)\n'
             'Patterns = model misfit  |  Funnel shape = heteroscedasticity',
             fontsize=13, fontweight='bold')

for col, (mname, yv, yp) in enumerate(model_preds):
    for row, (ti, tname) in enumerate(zip(range(2), TARGET_LABELS)):
        ax = axes[row, col]
        residuals = yv[:, ti] - yp[:, ti]
        ax.scatter(yp[:, ti], residuals, alpha=0.6, edgecolors='k',
                   linewidths=0.3, color='darkorange', s=40)
        ax.axhline(0, color='red', linestyle='--', lw=1.5)
        ax.set_title(f'{mname}  [{tname}]')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Residual (Actual − Predicted)')

plt.tight_layout()
plt.show()

print('\nDiagnostic complete.')
print('Interpretation guide:')
print('  Train/Val RMSE gap  >  2×  →  likely overfitting')
print('  Both train & val RMSE high  →  likely underfitting')
print('  Predicted vs Actual: points hugging the diagonal  →  good fit')
print('  Residuals: no pattern around zero  →  unbiased model')

## 5. Helper — Bias Score for High GU + Low cAMP

The **bias score** for a feature $f$ across samples is computed on **normalised** SHAP values:

$$\\hat{\\phi}_{gu}(f) = \\frac{\\overline{\\phi}_{gu}(f)}{\\max_j |\\overline{\\phi}_{gu}(j)|}, \\quad \\hat{\\phi}_{camp}(f) = \\frac{\\overline{\\phi}_{camp}(f)}{\\max_j |\\overline{\\phi}_{camp}(j)|}$$

$$\\text{BiasScore}(f) = \\hat{\\phi}_{gu}(f) - \\hat{\\phi}_{camp}(f)$$

**Why normalise?** GU and cAMP can have very different SHAP magnitude scales. Without normalisation the target with larger absolute SHAP values dominates the subtraction, obscuring the true feature preference. Dividing each target's mean SHAP vector by its own max-absolute value maps both onto $[-1, 1]$ before subtraction, so each target contributes equally to the score.

A positive bias score means the feature pushes toward **high GU AND low cAMP** — the desired combination.

In [ ]:
def bias_score_plot(shap_gu, shap_camp, feature_names, top_n=20, title='Bias Score'):
    """
    shap_gu, shap_camp : (N, F) SHAP value arrays

    Normalised bias score:
        mean_gu   is divided by max(|mean_gu|)   → range [-1, 1]
        mean_camp is divided by max(|mean_camp|) → range [-1, 1]
        bias = norm_gu - norm_camp

    This ensures both targets contribute equally regardless of their
    absolute SHAP magnitude scales.
    Positive → feature promotes high GU and/or low cAMP
    """
    mean_gu   = shap_gu.mean(axis=0)
    mean_camp = shap_camp.mean(axis=0)

    # ── NEW: per-target normalisation to [-1, 1] ──────────────────────────
    max_gu   = np.abs(mean_gu).max()
    max_camp = np.abs(mean_camp).max()
    norm_gu   = mean_gu   / max_gu   if max_gu   > 0 else mean_gu
    norm_camp = mean_camp / max_camp if max_camp > 0 else mean_camp
    bias = norm_gu - norm_camp  # range approx [-2, 2]
    # ──────────────────────────────────────────────────────────────────────

    idx_sorted = np.argsort(np.abs(bias))[::-1][:top_n]
    top_names  = [feature_names[i] for i in idx_sorted]
    top_bias   = bias[idx_sorted]

    colors = ['#2196F3' if b > 0 else '#F44336' for b in top_bias]
    plt.figure(figsize=(10, 6))
    plt.barh(top_names[::-1], top_bias[::-1], color=colors[::-1])
    plt.axvline(0, color='black', linewidth=0.8)
    plt.xlabel('Normalised Bias Score  (↑ favours High GU & Low cAMP)')
    plt.title(title)
    plt.tight_layout()
    plt.show()

    top_df = pd.DataFrame({
        'Feature':        top_names,
        'BiasScore':      top_bias,
        'NormSHAP_gu':    norm_gu[idx_sorted],
        'NormSHAP_camp':  norm_camp[idx_sorted],
        'RawMeanSHAP_gu':   mean_gu[idx_sorted],
        'RawMeanSHAP_camp': mean_camp[idx_sorted],
    })
    return top_df

print('Normalised bias-score helper defined.')


## 6. Explainable AI methods
### 6.1 KernelSHAP — SVR

KernelSHAP is model-agnostic and the appropriate SHAP method for SVR.

In [ ]:
# Use best fold (lowest mean RMSE across targets)
best_fold_svr = np.argmin([
    (m['gu']['RMSE'] + m['camp']['RMSE']) / 2 for m in svr_fold_metrics
])
scaler_svr, mo_svr_best, tr_idx, val_idx = svr_fold_models[best_fold_svr]

X_tr_sc_svr  = scaler_svr.transform(X_filtered[tr_idx])
X_val_sc_svr = scaler_svr.transform(X_filtered[val_idx])

# KernelSHAP uses a background summary (k-means to keep it tractable)
background = shap.kmeans(X_tr_sc_svr, 30)

shap_svr_gu, shap_svr_camp = [], []

for i, estimator in enumerate(mo_svr_best.estimators_):
    print(f'  Computing KernelSHAP for SVR output {i+1}/2 ...')
    explainer = shap.KernelExplainer(estimator.predict, background)
    sv = explainer.shap_values(X_val_sc_svr, nsamples=200)
    if i == 0:
        shap_svr_gu = sv
    else:
        shap_svr_camp = sv

# ── Individual SHAP summary plots ─────────────────────────────────────────
print('\n[SVR] SHAP Summary — GU:')
shap.summary_plot(shap_svr_gu, X_val_sc_svr, feature_names=names_filtered,
                  max_display=20, plot_type='bar', show=True)

print('\n[SVR] SHAP Summary — cAMP:')
shap.summary_plot(shap_svr_camp, X_val_sc_svr, feature_names=names_filtered,
                  max_display=20, plot_type='bar', show=True)

# ── Bias score ─────────────────────────────────────────────────────────────
print('\n[SVR] Bias Score — High GU & Low cAMP:')
svr_bias_df = bias_score_plot(
    np.array(shap_svr_gu), np.array(shap_svr_camp),
    names_filtered, top_n=20, title='SVR — Bias Score (High GU & Low cAMP)'
)
display(svr_bias_df.head(10))

### 6.2.1 TreeSHAP — Random Forest

In [ ]:
best_fold_rf = np.argmin([
    (m['gu']['RMSE'] + m['camp']['RMSE']) / 2 for m in rf_fold_metrics
])
rf_best, tr_idx_rf, val_idx_rf = rf_fold_models[best_fold_rf]

X_tr_rf  = X_filtered[tr_idx_rf]
X_val_rf = X_filtered[val_idx_rf]

# ── (a) MDI Feature Importance ─────────────────────────────────────────────
# RF natively stores multi-output importance as the mean over estimators/outputs
fi = rf_best.feature_importances_   # shape (F,)

# Per-output importance by examining each tree's per-output contributions
# (sklearn averages them internally, so both rows will be identical — shown for completeness)
n_feats = X_tr_rf.shape[1]
fi_per_output = np.zeros((2, n_feats))
for tree in rf_best.estimators_:
    fi_per_output += tree.feature_importances_.reshape(1, n_feats)
fi_per_output /= len(rf_best.estimators_)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, target_i, target_name in zip(axes, [0, 1], ['GU', 'cAMP']):
    top20_idx = np.argsort(fi_per_output[target_i])[::-1][:20]
    ax.barh([names_filtered[i] for i in top20_idx][::-1],
             fi_per_output[target_i][top20_idx][::-1], color='teal')
    ax.set_title(f'RF MDI Importance — {target_name}')
    ax.set_xlabel('Importance')
plt.suptitle('Random Forest Feature Importance (MDI)', fontweight='bold')
plt.tight_layout()
plt.show()

# ── (b) TreeSHAP ───────────────────────────────────────────────────────────
rf_explainer = shap.TreeExplainer(rf_best)
shap_rf_raw = rf_explainer.shap_values(X_val_rf)

# SHAP + multi-output RF can return:
#   • a list of 2 arrays, each (N, F)          → older SHAP / single-output path
#   • a single 3-D array  (N, F or F+1, 2)    → newer SHAP versions
# In either case we trim to exactly (N, n_feats) per output.
n_feats_val = X_val_rf.shape[1]

if isinstance(shap_rf_raw, list):
    # list of per-output arrays — trim any extra column (bias term)
    shap_rf_gu   = np.array(shap_rf_raw[0])[:, :n_feats_val]
    shap_rf_camp = np.array(shap_rf_raw[1])[:, :n_feats_val]
elif isinstance(shap_rf_raw, np.ndarray) and shap_rf_raw.ndim == 3:
    # shape (N, F_or_F+1, n_outputs) — slice features then outputs
    shap_rf_gu   = shap_rf_raw[:, :n_feats_val, 0]
    shap_rf_camp = shap_rf_raw[:, :n_feats_val, 1]
else:
    raise ValueError(
        f"Unexpected shap_values shape/type: "
        f"{type(shap_rf_raw)}, "
        f"{getattr(shap_rf_raw, 'shape', 'N/A')}"
    )

# Sanity-check before plotting
assert shap_rf_gu.shape == X_val_rf.shape, (
    f"Shape mismatch after trimming: SHAP {shap_rf_gu.shape} vs X {X_val_rf.shape}"
)

print('\n[RF] SHAP Beeswarm — GU:')
shap.summary_plot(shap_rf_gu, X_val_rf, feature_names=names_filtered, max_display=20)

print('\n[RF] SHAP Beeswarm — cAMP:')
shap.summary_plot(shap_rf_camp, X_val_rf, feature_names=names_filtered, max_display=20)

print('\n[RF] Bias Score — High GU & Low cAMP:')
rf_bias_df = bias_score_plot(
    shap_rf_gu, shap_rf_camp,
    names_filtered, top_n=20,
    title='Random Forest — Bias Score (High GU & Low cAMP)'
)
display(rf_bias_df.head(10))

### 6.2.2  Feature Importance — Random Forest

In [ ]:
best_fold_rf = np.argmin([
    (m['gu']['RMSE'] + m['camp']['RMSE']) / 2 for m in rf_fold_metrics
])
rf_best, tr_idx_rf, val_idx_rf = rf_fold_models[best_fold_rf]

X_tr_rf  = X_filtered[tr_idx_rf]
X_val_rf = X_filtered[val_idx_rf]

# ── (a) MDI Feature Importance ─────────────────────────────────────────────
# RF natively stores multi-output importance as the mean over estimators/outputs
fi = rf_best.feature_importances_   # shape (F,)

# Per-output importance by examining each tree's per-output contributions
# (sklearn averages them internally, so both rows will be identical — shown for completeness)
n_feats = X_tr_rf.shape[1]
fi_per_output = np.zeros((2, n_feats))
for tree in rf_best.estimators_:
    fi_per_output += tree.feature_importances_.reshape(1, n_feats)
fi_per_output /= len(rf_best.estimators_)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, target_i, target_name in zip(axes, [0, 1], ['GU', 'cAMP']):
    top20_idx = np.argsort(fi_per_output[target_i])[::-1][:20]
    ax.barh([names_filtered[i] for i in top20_idx][::-1],
             fi_per_output[target_i][top20_idx][::-1], color='teal')
    ax.set_title(f'RF MDI Importance — {target_name}')
    ax.set_xlabel('Importance')
plt.suptitle('Random Forest Feature Importance (MDI)', fontweight='bold')
plt.tight_layout()
plt.show()

# ── (b) TreeSHAP ───────────────────────────────────────────────────────────
rf_explainer = shap.TreeExplainer(rf_best)
shap_rf_raw = rf_explainer.shap_values(X_val_rf)

# SHAP + multi-output RF can return:
#   • a list of 2 arrays, each (N, F)          → older SHAP / single-output path
#   • a single 3-D array  (N, F or F+1, 2)    → newer SHAP versions
# In either case we trim to exactly (N, n_feats) per output.
n_feats_val = X_val_rf.shape[1]

if isinstance(shap_rf_raw, list):
    # list of per-output arrays — trim any extra column (bias term)
    shap_rf_gu   = np.array(shap_rf_raw[0])[:, :n_feats_val]
    shap_rf_camp = np.array(shap_rf_raw[1])[:, :n_feats_val]
elif isinstance(shap_rf_raw, np.ndarray) and shap_rf_raw.ndim == 3:
    # shape (N, F_or_F+1, n_outputs) — slice features then outputs
    shap_rf_gu   = shap_rf_raw[:, :n_feats_val, 0]
    shap_rf_camp = shap_rf_raw[:, :n_feats_val, 1]
else:
    raise ValueError(
        f"Unexpected shap_values shape/type: "
        f"{type(shap_rf_raw)}, "
        f"{getattr(shap_rf_raw, 'shape', 'N/A')}"
    )

# Sanity-check before plotting
assert shap_rf_gu.shape == X_val_rf.shape, (
    f"Shape mismatch after trimming: SHAP {shap_rf_gu.shape} vs X {X_val_rf.shape}"
)

print('\n[RF] SHAP Beeswarm — GU:')
shap.summary_plot(shap_rf_gu, X_val_rf, feature_names=names_filtered, max_display=20)

print('\n[RF] SHAP Beeswarm — cAMP:')
shap.summary_plot(shap_rf_camp, X_val_rf, feature_names=names_filtered, max_display=20)

print('\n[RF] Bias Score — High GU & Low cAMP:')
rf_bias_df = bias_score_plot(
    shap_rf_gu, shap_rf_camp,
    names_filtered, top_n=20,
    title='Random Forest — Bias Score (High GU & Low cAMP)'
)
display(rf_bias_df.head(10))

### 6.3 TreeSHAP + Feature Importance — XGBoost

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import shap

# ─────────────────────────────────────────────────────────────────────────────
# Select best fold
# ─────────────────────────────────────────────────────────────────────────────
best_fold_xgb = np.argmin([
    (m['gu']['RMSE'] + m['camp']['RMSE']) / 2
    for m in xgb_fold_metrics
])

xgb_best, tr_idx_xgb, val_idx_xgb = xgb_fold_models[best_fold_xgb]

# IMPORTANT FIX: use X_filtered (same features used for training)
X_tr_xgb  = X_filtered[tr_idx_xgb]
X_val_xgb = X_filtered[val_idx_xgb]

# ─────────────────────────────────────────────────────────────────────────────
# (a) SHAP-Based Feature Importance (KernelExplainer for multi-output XGBoost)
# ─────────────────────────────────────────────────────────────────────────────

def predict_gu(x):
    return xgb_best.predict(x)[:, 0]

def predict_camp(x):
    return xgb_best.predict(x)[:, 1]

# Background dataset must match model feature shape
bg = shap.kmeans(X_tr_xgb, 30)

print("Computing SHAP values for GU (this may take a moment)...")
ex_gu = shap.KernelExplainer(predict_gu, bg)
shap_xgb_gu = ex_gu.shap_values(X_val_xgb, nsamples=200)

print("Computing SHAP values for cAMP...")
ex_camp = shap.KernelExplainer(predict_camp, bg)
shap_xgb_camp = ex_camp.shap_values(X_val_xgb, nsamples=200)

# Compute mean absolute SHAP importance across both targets
fi_xgb = (
    np.abs(shap_xgb_gu).mean(axis=0) +
    np.abs(shap_xgb_camp).mean(axis=0)
) / 2

top10_xgb = np.argsort(fi_xgb)[::-1][:10]

plt.figure(figsize=(10, 6))
plt.barh(
    [names_filtered[i] for i in top10_xgb][::-1],
    fi_xgb[top10_xgb][::-1]
)
plt.xlabel("Mean |SHAP value|")
plt.title("XGBoost Feature Importance (Multi-Output, SHAP-Based)")
plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# (b) SHAP Beeswarm Plots
# ─────────────────────────────────────────────────────────────────────────────
print('\n[XGB] SHAP Beeswarm — GU:')
shap.summary_plot(
    shap_xgb_gu,
    X_val_xgb,
    feature_names=names_filtered,
    max_display=20
)

print('\n[XGB] SHAP Beeswarm — cAMP:')
shap.summary_plot(
    shap_xgb_camp,
    X_val_xgb,
    feature_names=names_filtered,
    max_display=20
)

# ─────────────────────────────────────────────────────────────────────────────
# (c) Bias Score Analysis
# ─────────────────────────────────────────────────────────────────────────────
print('\n[XGB] Bias Score — High GU & Low cAMP:')
xgb_bias_df = bias_score_plot(
    shap_xgb_gu,
    shap_xgb_camp,
    names_filtered,
    top_n=20,
    title='XGBoost — Bias Score (High GU & Low cAMP)'
)
display(xgb_bias_df.head(10))


## 7. Consensus Bias Score Across Models

Aggregate MACCS-level bias scores from SVR, RF, and XGBoost to find features that **all models agree** drive high GU + low cAMP.

In [ ]:
def get_bias_vector(shap_gu, shap_camp):
    """
    Normalised bias vector: divides each target's mean SHAP by its own
    max-absolute value before subtracting, so both targets are on [-1,1].
    """
    mean_gu   = shap_gu.mean(axis=0)
    mean_camp = shap_camp.mean(axis=0)
    max_gu    = np.abs(mean_gu).max()
    max_camp  = np.abs(mean_camp).max()
    norm_gu   = mean_gu   / max_gu   if max_gu   > 0 else mean_gu
    norm_camp = mean_camp / max_camp if max_camp > 0 else mean_camp
    return norm_gu - norm_camp

bias_svr  = get_bias_vector(np.array(shap_svr_gu),   np.array(shap_svr_camp))
bias_rf   = get_bias_vector(shap_rf_gu,   shap_rf_camp)
bias_xgb  = get_bias_vector(shap_xgb_gu,  shap_xgb_camp)

# Each bias vector is already normalised per-target; scale to [-1,1] across
# features before averaging across models so no single model dominates.
def normalise(v):
    m = np.abs(v).max()
    return v / m if m > 0 else v

consensus_bias = (normalise(bias_svr) + normalise(bias_rf) + normalise(bias_xgb)) / 3.0

top_n = 25
idx_top = np.argsort(np.abs(consensus_bias))[::-1][:top_n]
names_top = [names_filtered[i] for i in idx_top]
vals_top  = consensus_bias[idx_top]

colors = ['#1976D2' if v > 0 else '#D32F2F' for v in vals_top]
plt.figure(figsize=(11, 7))
plt.barh(names_top[::-1], vals_top[::-1], color=colors[::-1])
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Normalised Consensus Bias Score  (range ≈ −1 to +1)')
plt.title('Consensus Bias Score (SVR + RF + XGBoost)\n↑ = High Glucose Uptake & Low cAMP', fontsize=13)
plt.tight_layout()
plt.show()

consensus_df = pd.DataFrame({
    'Feature':        names_top,
    'ConsensusScore': vals_top,
    'SVR_bias':  bias_svr[idx_top],
    'RF_bias':   bias_rf[idx_top],
    'XGB_bias':  bias_xgb[idx_top]
}).sort_values('ConsensusScore', ascending=False)

print('Top features driving HIGH GU & LOW cAMP (positive score):')
display(consensus_df[consensus_df['ConsensusScore'] > 0].head(10))
print('\nTop features that hinder the desired outcome (negative score):')
display(consensus_df[consensus_df['ConsensusScore'] < 0].head(10))


## 8. MACCS Key Interpretation Helper

Map top-scoring MACCS bit indices back to their chemical meaning.

In [ ]:
# ============================================================
# 1. FULL MACCS KEY DESCRIPTION DICTIONARY
#    (Your original entries + all missing bits you requested)
# ============================================================

MACCS_DESCRIPTIONS = {
    1: 'ISOTOPE',
    2: 'Atomic num > 103 (rare element)',
    3: 'Group IVA, VA, VIA periods 4+ (Ge, Sn, Pb, As, Sb, Bi, Se, Te)',
    4: 'Actinide',
    5: 'Atom with atomic num > 103',
    6: 'Group IIIB,IVB (Sc, Ti, La, Hf, ...)',
    7: 'Lanthanide',
    8: 'Group VB, VIB, VIIB (V, Cr, Mn, Nb, Mo, Tc, Ta, W, Re)',
    9: 'QAAA@1 (4-atom chain)',
    10: 'Group VIII (Fe, Co, Ni, Ru, Rh, Pd, Os, Ir, Pt)',
    11: 'Group IIA (Alkaline earth: Be, Mg, Ca, Sr, Ba, Ra)',
    12: '4-membered ring with heteroatom',
    13: 'Group IB, IIB (Cu, Zn, Ag, Cd, Au, Hg)',
    14: 'Aromatic ring (A@1)',
    15: 'Halide bonded to heteroatom',
    16: 'Nitrogen with single bond to oxygen (N-O)',
    17: 'Atom is in a 3-membered ring',
    18: 'Group IA (Alkali metals: Li, Na, K, Rb, Cs, Fr)',
    19: 'QHAQH (heteroatom-carbon chain pattern)',
    20: 'Aromatic ring (size >= 5)',
    21: 'Heteroatom in 3-membered ring',
    22: 'NH2 or OH (primary amine or alcohol)',
    23: 'Charged atom (Nitrogen positive or anion)',
    24: 'S (sulfur present)',
    25: 'Amide or thioamide N-C=O/S',
    26: 'Aromatic C-S bond',
    27: 'Nitrogen with 3 bonds, not N+',
    28: '2-ring system fused',
    29: 'Bridged or polycyclic',
    30: 'Atom in ring of size >= 4',
    31: 'F, Cl, Br, I (any halogen)',
    32: 'Si (silicon present)',
    33: 'B (boron present)',
    34: 'Sn or Pb (tin or lead)',
    35: 'D (deuterium / isotope label)',
    36: 'Aryl iodide',
    37: 'Phosphorus-oxygen bond (P-O)',
    38: 'Aryl bromide',
    39: 'Heteroatom (not C, H)',
    40: 'Quaternary carbon (C with 4 non-H)',
    41: 'Aromatic C-Cl',
    42: 'Ester (C(=O)O or OC=O)',
    43: 'Trifluoromethyl (-CF3)',
    44: 'Any ring',
    45: 'Halogen-aliphatic C-Cl',
    46: 'Nitro group (-NO2)',
    47: 'N-oxide (N->O)',
    48: 'Aromatic C-F',
    49: 'Imine or oxime (C=N)',
    50: 'Nitrile (-C#N)',
    51: 'Diazo, azide or N=N',
    52: 'Alkyl chloride (Cl-C)',
    53: 'Isocyanate or isothiocyanate (N=C=O or N=C=S)',
    54: 'Cyclopropane or cyclobutane ring',
    55: 'Alkyl iodide (I-C)',
    56: 'P presence',
    57: 'Sulfonyl (S(=O)=O)',
    58: 'Aromatic',
    59: 'Epoxide',
    60: 'Aryl fluoride',
    61: 'Nitrogen heterocycle (N in ring)',
    62: 'Aldehyde (C=O, H on carbonyl C)',
    63: 'QHAAAQH (pattern)',
    64: 'Alkene (C=C, non-aromatic)',
    65: 'Nitrogen-heteroatom ring',
    66: 'S heterocycle (S in ring)',
    67: 'Phosphate ester (O-P=O)',
    68: 'Se or Te',
    69: 'Alpha-beta unsaturated carbonyl (C=C-C=O)',
    70: 'Halogen on a non-aromatic ring',
    71: 'Hydrazine or hydrazone (N-N)',
    72: 'Azo (Ar-N=N-Ar)',
    73: 'Quaternary nitrogen (N+)',
    74: 'Vinyl halide (X-C=C)',
    75: 'Pyridine ring',
    76: 'Two C=O in molecule',
    77: 'Carbamate (N-C(=O)-O)',
    78: 'Two rings sharing bond (fused)',
    79: 'OH attached to aromatic ring',
    80: 'Guanidine (N-C(=N))',
    81: 'Phosphoric acid or ester',
    82: 'Atom is part of a heteroatom–heteroatom bond',
    83: 'C=O (ketone or aldehyde)',
    84: 'Acrylate / acryloyl (C=CC=O)',
    85: 'Beta-lactam (4-ring with N-C=O)',
    86: 'Sulfone (C-S(=O)(=O)-C)',
    87: 'Amidine (C(=N)N)',
    88: 'Secondary aliphatic amine (NH between 2 C)',
    89: 'Carboxylic acid anhydride (C(=O)OC=O)',
    90: 'Alkyne (C#C)',
    91: 'Atom is in a fused ring system',
    92: 'Oxygen in ring (ether oxygen in ring)',
    93: 'Atom is in a 5-membered heterocycle',
    94: 'Nitrogen in 6-membered ring',
    95: 'Atom is in a 6-membered heterocycle',
    96: 'Oxygen in 5-membered ring',
    97: 'Oxygen in 6-membered ring',
    98: 'Nitrogen in 5-membered ring',
    99: 'N-C-N pattern',
    100: 'Quaternary N',
    101: 'Aromatic C-N bond',
    102: 'HBA (hydrogen bond acceptor)',
    103: 'HBD (hydrogen bond donor)',
    104: 'Phenyl (benzene substituent)',
    105: 'Charged molecule (ionic)',
    106: 'Ketone (C-C(=O)-C)',
    107: 'OH or NH',
    108: 'Oxygen or Sulfur present',
    109: 'Sulfoxide (C-S(=O)-C)',
    110: 'Aromatic nitrogen',
    111: 'Aromatic ring (fused)',
    112: 'Two aromatic rings (biphenyl-like)',
    113: 'Aromatic heterocycle',
    114: 'Aromatic ring with N',
    115: 'Aromatic ring with O or S',
    116: 'Ether (C-O-C)',
    117: 'Tertiary aliphatic amine (N with 3 C)',
    118: 'Secondary amine (NH)',
    119: 'Aliphatic OH (non-phenol)',
    120: 'Sulfide (C-S-C)',
    121: 'OH group',
    122: 'NH group',
    123: 'C=C (alkene)',
    124: 'Amide (N-C=O)',
    125: 'Thiol (SH)',
    126: 'Carbonyl attached to heteroatom',
    127: 'Benzene ring',
    128: 'Cyclopentane ring',
    129: 'Cyclohexane ring',
    130: 'C=O (any carbonyl)',
    131: 'N-H (any)',
    132: 'Nitrogen adjacent to ring',
    133: 'Fluorine present',
    134: 'Nitrogen-containing ring',
    135: 'Double bond (any)',
    136: 'Ring junction',
    137: 'Aromatic atom count >=2',
    138: 'Two heteroatoms',
    139: 'Halogen present',
    140: 'Aromatic C-O bond',
    141: 'Aromatic C=O',
    142: 'Aromatic amine (Ar-NH2 or Ar-NH)',
    143: 'NH2 group',
    144: 'Halogen attached to aromatic ring',
    145: 'Phenol / OH on aromatic ring',
    146: 'C-O (single bond)',
    147: 'Aromatic ring',
    148: 'Methyl group',
    149: 'Nitrogen (any)',
    150: 'Branching (quaternary C or branch point)',
    151: 'Two nitrogen atoms',
    152: 'Two oxygen atoms',
    153: 'Tertiary N (not aromatic)',
    154: 'Ester (C(=O)O)',
    155: 'Carboxylic acid',
    156: 'Sulfonamide or sulfonyl group',
    157: 'Chlorine present',
    158: 'Two rings',
    159: 'Oxygen (any)',
    160: 'Primary amine',
    161: 'N-C=O pattern',
    162: 'Aromatic C-C',
    163: 'Hydroxyl (OH)',
    164: 'Bromine present',
    165: 'Nitrogen present',
    166: 'Aromatic oxygen',
}

# ============================================================
# 2. Helper function to map MACCS bit → description
# ============================================================

def describe_maccs(name):
    idx = int(name.split('_')[-1])
    return MACCS_DESCRIPTIONS.get(idx, f'Bit {idx} (no description)')


# ============================================================
# 3. Apply descriptions to your consensus dataframe
# ============================================================

consensus_df['ChemicalMeaning'] = consensus_df['Feature'].apply(describe_maccs)


# ============================================================
# 4. Display top positive and negative bias features
# ============================================================

print('Top positive bias features (High GU & Low cAMP):')
display(
    consensus_df[consensus_df['ConsensusScore'] > 0]
    [['Feature','ChemicalMeaning','ConsensusScore']]
    .sort_values('ConsensusScore', ascending=False)
    .head(10)
)

print('\nTop negative bias features:')
display(
    consensus_df[consensus_df['ConsensusScore'] < 0]
    [['Feature','ChemicalMeaning','ConsensusScore']]
    .sort_values('ConsensusScore')
    .head(10)
)
